# 03 — Preparação de features

Transforma as 61 colunas do parquet processado na matriz que entra nos três modelos
exigidos pelo ADR-004, aplicando as decisões que a EDA levantou mas não implementou.

A análise exploratória está completa em `notebooks/02_eda.ipynb`, 15 seções. Este
notebook não a refaz: consome as conclusões de lá e decide o que fazer com elas.

Também não faz limpeza de dados no sentido usual. A seção 12 da EDA verificou zero
duplicatas, zero valores fora de domínio e validou a base contra um contrato `pandera`.
O que existe são nulos e zeros com significado de negócio, e a seção 3 mostra o que um
`dropna()` faria com eles.

A lógica mora em `src/features/`. Aqui só se consome `build_pipeline()` e se registra a
evidência de cada decisão, para que exista uma implementação só, compartilhada com a API.

Rodar de cima para baixo, sem pular célula.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

# E402: o sys.path.insert acima precisa rodar antes destes imports,
# caso contrario `src` nao e encontrado ao abrir o notebook direto.
from src.features import config as cfg  # noqa: E402
from src.features.preparation import (  # noqa: E402
    EngenhariaEstrutural,
    build_pipeline,
    filtrar_censura,
    separar_alvo,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

SEED = 57
CAMINHO = RAIZ / "data" / "processed" / "telco_churn_processed.parquet"

if not CAMINHO.exists():
    raise FileNotFoundError(
        f"{CAMINHO} nao encontrado. Rode antes: uv run python -m src.data.pipeline"
    )

df = pd.read_parquet(CAMINHO)
print(f"Base carregada: {df.shape[0]} linhas x {df.shape[1]} colunas")

Base carregada: 7043 linhas x 61 colunas


A EDA reporta 7043 linhas por 61 colunas. Número diferente significa que o ETL mudou, e
aí as listas em `src/features/config.py` precisam ser conferidas antes de seguir.

In [2]:
taxa = df[cfg.ALVO].mean()
print(f"Taxa de churn: {taxa:.1%}  ({int(df[cfg.ALVO].sum())} clientes)")
print(f"Acuracia de um modelo que preve 'ninguem cancela': {1 - taxa:.1%}")
print()
print(df[cfg.COLUNA_STATUS].value_counts().to_string())

Taxa de churn: 26.5%  (1869 clientes)
Acuracia de um modelo que preve 'ninguem cancela': 73.5%

status_customer_status
Stayed     4720
Churned    1869
Joined      454


Guarde o segundo número. Com 26,5% de churn, prever que ninguém cancela já acerta 73,5%,
e qualquer acurácia reportada sem esse piso ao lado não diz nada. O bloco 6 do ML Canvas,
que define a métrica de avaliação, está em branco. Proposta na seção 9.

## 2. Inventário

61 colunas, prefixadas com a base de origem (`demographics_`, `locations_`, `services_`,
`status_`, `populations_`), exceto as chaves de junção.

In [3]:
inventario = (
    pd.DataFrame(
        {
            "coluna": df.columns,
            "tipo": [str(t) for t in df.dtypes],
            "nulos_%": (df.isna().mean() * 100).round(1).values,
            "valores_unicos": [df[c].nunique(dropna=True) for c in df.columns],
        }
    )
    .assign(origem=lambda d: d["coluna"].str.split("_").str[0])
    .sort_values(["origem", "coluna"])
    .reset_index(drop=True)
)

print(inventario.groupby("origem").size().to_string())
inventario.head(20)

origem
customer         1
demographics     8
locations        8
populations      2
services        30
status          11
zip              1


,coluna,tipo,nulos_%,valores_unicos,origem
0,customer_id,object,0.0,7043,customer
1,demographics_age,int64,0.0,62,demographics
2,demographics_count,int64,0.0,1,demographics
3,demographics_dependents,object,0.0,2,demographics
4,demographics_gender,object,0.0,2,demographics
5,demographics_married,object,0.0,2,demographics
6,demographics_number_of_dependents,int64,0.0,10,demographics
7,demographics_senior_citizen,object,0.0,2,demographics
8,demographics_under_30,object,0.0,2,demographics
9,locations_city,object,0.0,1106,locations


## 3. Nulos e zeros que carregam significado

Quatro colunas concentram praticamente todos os nulos da base, e em nenhuma delas o nulo
representa dado faltante.

In [4]:
nulos = df.isna().mean().mul(100).round(1).loc[lambda s: s > 0].sort_values(ascending=False)
nulos.to_frame("nulos_%")

,nulos_%
status_churn_reason,73.5
status_churn_category,73.5
services_offer,55.0
services_internet_type,21.7


In [5]:
linhas_apos_dropna = len(df.dropna())
print(f"Base original: {len(df):>5} linhas")
print(f"Apos dropna(): {linhas_apos_dropna:>5} linhas")
print(f"Perda:         {1 - linhas_apos_dropna / len(df):>5.1%}")
print()

sem_internet = df["services_internet_type"].isna()
print("services_internet_type nulo <=> cliente sem internet:")
print(df.loc[sem_internet, "services_internet_service"].value_counts().to_string())
print()
print(f"Churn de quem NAO tem internet: {df.loc[sem_internet, cfg.ALVO].mean():.1%}")
print(f"Churn de quem TEM internet:     {df.loc[~sem_internet, cfg.ALVO].mean():.1%}")

Base original:  7043 linhas
Apos dropna():   757 linhas
Perda:         89.3%

services_internet_type nulo <=> cliente sem internet:
services_internet_service
No    1526

Churn de quem NAO tem internet: 7.4%
Churn de quem TEM internet:     31.8%


Não ter internet é o fator de proteção mais forte da base inteira: 7,4% de churn contra
31,8%. Um `dropna()` apagaria exatamente esse grupo, e um `fillna("Desconhecido")` o
misturaria com dado genuinamente ausente. A ausência vira categoria própria,
`"No Internet Service"`, que é o que `EngenhariaEstrutural` faz. Rótulo em inglês de
propósito: o resto do domínio categórico da coluna (`"Cable"`, `"DSL"`, `"Fiber Optic"`)
já vem em inglês da IBM, e um rótulo em português destoaria.

Vale o mesmo para `services_offer`, onde 55% de nulos significam cliente sem oferta
promocional (`"No Offer"`). Já `status_churn_reason` e `status_churn_category` são
vazamento e saem por outro motivo, tratado na seção 5.1.

Nos zeros o raciocínio se repete. `avg_monthly_gb_download == 0` identifica quem não tem
internet, não quem consome pouco. Escalonar a coluna sem separar os dois grupos faz o
`StandardScaler` calcular média e desvio sobre duas populações empilhadas.

In [6]:
for coluna, flag in cfg.ZEROS_ESTRUTURAIS.items():
    if coluna not in df.columns:
        print(f"[ausente] {coluna}")
        continue
    eh_zero = df[coluna] == 0
    print(f"{coluna}")
    print(f"  zeros: {eh_zero.sum():>5} ({eh_zero.mean():.1%})  ->  vira '{flag}'")
    print(f"  media entre os nao-zero: {df.loc[~eh_zero, coluna].mean():.2f}")
    print(f"  churn | zero:     {df.loc[eh_zero, cfg.ALVO].mean():.1%}")
    print(f"  churn | nao-zero: {df.loc[~eh_zero, cfg.ALVO].mean():.1%}")
    print()

services_avg_monthly_gb_download
  zeros:  1526 (21.7%)  ->  vira 'flag_sem_internet'
  media entre os nao-zero: 26.19
  churn | zero:     7.4%
  churn | nao-zero: 31.8%

services_avg_monthly_long_distance_charges
  zeros:   682 (9.7%)  ->  vira 'flag_sem_telefone'
  media entre os nao-zero: 25.42
  churn | zero:     24.9%
  churn | nao-zero: 26.7%



## 4. Os clientes `Joined`

454 clientes (6,4%) têm status `Joined`. Entraram há 1 a 3 meses e estão rotulados como
não-churn.

A leitura inicial foi de censura à direita: eles não teriam tido tempo de cancelar, e
mantê-los rotulados como 0 atenuaria o efeito de tenure, que é o segundo preditor mais
forte segundo a EDA. A conclusão foi remover.

A base contradiz essa leitura.

In [7]:
joined = df[cfg.COLUNA_STATUS].astype(str).str.strip().str.casefold() == "joined"
baixo = df["services_tenure_in_months"] <= 3

print("Distribuicao de tenure entre os Joined (meses):")
print(df.loc[joined, "services_tenure_in_months"].describe().round(1).to_string())
print()
print("Status de TODOS os clientes com tenure <= 3:")
print(df.loc[baixo, cfg.COLUNA_STATUS].value_counts().to_string())
print()
churns_baixo = (baixo & ~joined).sum()
print(f"Nao-Joined nessa faixa: {churns_baixo}")
print(f"Equivale a {churns_baixo / df[cfg.ALVO].sum():.1%} de todos os churns da base")

Distribuicao de tenure entre os Joined (meses):
count    454.0
mean       1.7
std        0.8
min        1.0
25%        1.0
50%        1.0
75%        2.0
max        3.0

Status de TODOS os clientes com tenure <= 3:
status_customer_status
Churned    597
Joined     454

Nao-Joined nessa faixa: 597
Equivale a 31.9% de todos os churns da base


Na faixa tenure ≤ 3 existem 597 `Churned` e 454 `Joined`, e nenhum `Stayed`. As duas
categorias cobrem a faixa inteira, o que revela o que `Joined` de fato é: o rótulo que a
IBM dá ao cliente recente que ainda não cancelou. É a mesma condição de `Stayed`, com
menos tempo de casa, e como negativo dentro da janela observada ele é legítimo.

Remover tem custo próprio. Os 597 restantes passam a ser 100% de churn naquela faixa,
uma separação perfeita que o modelo aprende como regra determinística, e que é artefato
de rotulagem. Como esses 597 são 31,9% de todos os cancelamentos, o efeito na métrica não
é desprezível. A célula abaixo mede.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split


def avaliar_cenario(remover_joined):
    base_mod, _ = filtrar_censura(df, remover_joined=remover_joined)
    Xa, ya = separar_alvo(base_mod)
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
        Xa, ya, test_size=0.2, random_state=SEED, stratify=ya
    )
    saida = []
    for nome, est in {
        "regressao logistica": LogisticRegression(max_iter=1000, random_state=SEED),
        "random forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    }.items():
        p = build_pipeline(est).fit(Xa_tr, ya_tr)
        pb = p.predict_proba(Xa_te)[:, 1]
        saida.append(
            {
                "cenario": "sem Joined" if remover_joined else "com Joined",
                "modelo": nome,
                "prevalencia": ya.mean(),
                "roc_auc": roc_auc_score(ya_te, pb),
                "pr_auc": average_precision_score(ya_te, pb),
            }
        )
    return saida


pd.DataFrame(avaliar_cenario(True) + avaliar_cenario(False)).round(4).set_index(
    ["cenario", "modelo"]
)

prevalencia  roc_auc  pr_auc
cenario    modelo                                           
sem Joined regressao logistica       0.2837   0.9271  0.8403
           random forest             0.2837   0.9349  0.8778
com Joined regressao logistica       0.2654   0.9127  0.7800
           random forest             0.2654   0.9118  0.7861

Comparar apenas o ROC-AUC. O PR-AUC cai no cenário "com Joined" porque a prevalência sai
de 28,4% para 26,5%, e a métrica é sensível a isso; não é degradação do modelo.

Remover os `Joined` entrega cerca de 2 pontos de ROC-AUC que o modelo não teria em
produção. Decisão adotada: manter, e registrar o achado dos 597 como limitação no Model
Card. `filtrar_censura` tem `remover_joined=False` como padrão e aceita `True` para
reverter.

A objeção da censura volta a valer se o grupo definir o alvo como janela futura
("cancela nos próximos N meses") em vez do snapshot atual. Isso está em aberto e aparece
na seção 10.

In [9]:
modelagem, censurados = filtrar_censura(df)

print(f"Conjunto de modelagem: {len(modelagem)} linhas")
print(f"Recorte censurado:     {len(censurados)} linhas")
print(f"Taxa de churn: {modelagem[cfg.ALVO].mean():.1%}")

Conjunto de modelagem: 7043 linhas
Recorte censurado:     0 linhas
Taxa de churn: 26.5%


## 5. O que sai e por quê

Sete categorias de descarte, cada uma rastreável a uma seção da EDA ou a uma verificação
feita aqui.

In [10]:
categorias = {
    "constantes (variancia zero)": cfg.COLS_CONSTANTES,
    "identificadoras": cfg.COLS_IDENTIFICADORAS,
    "vazamento": cfg.COLS_VAZAMENTO,
    "derivadas do alvo": cfg.COLS_DERIVADAS_DO_ALVO,
    "redundantes": cfg.COLS_REDUNDANTES,
    "geografia (desligavel)": cfg.COLS_GEOGRAFIA,
    "quarentena (desligavel)": cfg.COLS_QUARENTENA,
}

for nome, colunas in categorias.items():
    presentes = [c for c in colunas if c in df.columns]
    ausentes = [c for c in colunas if c not in df.columns]
    print(f"{nome}: {len(presentes)}")
    for c in presentes:
        print(f"    {c}")
    for c in ausentes:
        print(f"    [NAO ENCONTRADA NA BASE] {c}")
    print()

constantes (variancia zero): 8
    services_quarter
    status_quarter
    locations_state
    locations_country
    locations_count
    demographics_count
    services_count
    status_count

identificadoras: 5
    customer_id
    locations_location_id
    services_service_id
    status_status_id
    populations_id

vazamento: 5
    status_satisfaction_score
    status_churn_score
    status_churn_category
    status_churn_reason
    status_cltv

derivadas do alvo: 2
    status_churn_label
    status_customer_status

redundantes: 5
    demographics_under_30
    services_internet_service
    services_referred_a_friend
    services_total_revenue
    services_total_long_distance_charges

geografia (desligavel): 6
    locations_city
    locations_lat_long
    zip_code
    locations_latitude
    locations_longitude
    populations_population

quarentena (desligavel): 1
    services_offer



Qualquer linha marcada com `[NAO ENCONTRADA NA BASE]` indica que a convenção de nomes do
ETL divergiu de `src/features/config.py`, e o config precisa ser corrigido antes de
seguir. O descarte é silencioso de propósito, porque o mesmo pipeline atende o payload do
`POST /predict`, que nunca carrega colunas `status_*`; sem esse aviso, um nome errado
passaria despercebido.

As redundâncias resolvidas: `demographics_under_30` é `age < 30`;
`services_internet_service` é `internet_type` não nulo; `services_referred_a_friend` é
`number_of_referrals > 0`; `services_total_revenue` e
`services_total_long_distance_charges` são identidades contábeis exatas das demais
colunas de cobrança.

### 5.1 Vazamento

A EDA identificou cinco colunas indisponíveis no momento da previsão. Rodando o pipeline
apareceram mais duas que não constavam do Contrato 1 nem do `eda-findings.md`:
`status_churn_label` e `status_customer_status` são reescritas literais do alvo.

A célula mede o custo de deixar cada uma passar.

In [11]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

X_bruto, y_bruto = separar_alvo(modelagem)


def auc_de(colunas):
    dados = X_bruto[colunas].copy()
    for c in dados.columns:
        if pd.api.types.is_numeric_dtype(dados[c]):
            dados[c] = dados[c].fillna(dados[c].median())
        else:
            dados[c] = dados[c].astype("object").fillna("Nulo").astype(str)
    pipe = Pipeline(
        [
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ("lr", LogisticRegression(max_iter=1000, random_state=SEED)),
        ]
    )
    return cross_val_score(pipe, dados, y_bruto, cv=5, scoring="roc_auc").mean()


for coluna in cfg.COLS_VAZAMENTO + cfg.COLS_DERIVADAS_DO_ALVO:
    if coluna in X_bruto.columns:
        print(f"AUC usando SO {coluna}: {auc_de([coluna]):.4f}")

AUC usando SO status_satisfaction_score: 0.9503
AUC usando SO status_churn_score: 0.9405


AUC usando SO status_churn_category: 1.0000


AUC usando SO status_churn_reason: 1.0000


AUC usando SO status_cltv: 0.5103
AUC usando SO status_churn_label: 1.0000
AUC usando SO status_customer_status: 1.0000


AUC de 1,0 com uma coluna só é a assinatura do vazamento. O `churn_score` para em 0,94
porque é um score e não o rótulo, mas é saída de um modelo anterior treinado sobre o mesmo
alvo, com dados que o grupo não tem.

Esse 0,94 vira o número a bater. Se o modelo do grupo não superar o `churn_score` usando
apenas features legítimas, ele não acrescenta nada ao que a IBM já entregou pronto.

Sobre a proposta de usar `churn_score` como input via stacking, debate já registrado no
`eda-findings.md`: vazamento diz respeito à disponibilidade do dado no momento da
previsão, não ao algoritmo que o consome. Stacking seria defensável se `churn_score` fosse
score de terceiros, recalculado de forma independente para todo cliente ativo e disponível
antes da nossa previsão, como um score de crédito externo. A proveniência é desconhecida e
o dado só existe neste snapshot.

O `status_cltv` fica em 0,51, praticamente aleatório. Sai mesmo assim, por ser estimativa
pré-calculada de proveniência desconhecida.

### 5.2 `services_offer`

A EDA registrou Offer E com 52,9% de churn contra Offer A com 6,7%, e concluiu que certas
campanhas atraem o perfil que cancela.

A hipótese inversa não foi checada. Se as ofertas forem ações de retenção aplicadas a
clientes já sinalizados como em risco, a causalidade se inverte e a coluna carrega
informação posterior ao sinal, o que a torna vazamento.

Sem o dicionário da IBM esclarecendo quando a oferta é atribuída, a coluna fica de fora e
`build_pipeline(incluir_offer=True)` reativa depois da checagem.

In [12]:
if "services_offer" in df.columns:
    resumo = (
        df.assign(oferta=df["services_offer"].fillna("No Offer"))
        .groupby("oferta")
        .agg(n=(cfg.ALVO, "size"), churn=(cfg.ALVO, "mean"))
        .assign(churn=lambda d: (d["churn"] * 100).round(1))
        .sort_values("churn", ascending=False)
    )
    display(resumo)

,n,churn
oferta,,
Offer E,805,52.9
No Offer,3877,27.1
Offer D,602,26.7
Offer C,415,22.9
Offer B,824,12.3
Offer A,520,6.7


O N aparece ao lado de cada taxa porque a EDA ressalvou San Diego (64,9% em 285 clientes)
e não aplicou o mesmo critério às ofertas. Aqui as faixas são grandes o bastante, entre
415 e 3877 clientes, então as taxas se sustentam.

## 6. Split antes da preparação

Dividir primeiro, ajustar depois. `StandardScaler` e `SimpleImputer` aprendem parâmetros
dos dados, e se enxergarem o conjunto de teste a métrica sai otimista. O README cita esse
bug como `fix/scaler-leak`.

Por isso a preparação vive dentro de um `Pipeline`: o `fit` roda uma vez, sobre o treino,
e o `transform` se aplica ao teste sem reajuste.

In [13]:
X, y = separar_alvo(modelagem)

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Treino: {len(X_treino)} linhas  |  churn {y_treino.mean():.1%}")
print(f"Teste:  {len(X_teste)} linhas  |  churn {y_teste.mean():.1%}")

Treino: 5634 linhas  |  churn 26.5%
Teste:  1409 linhas  |  churn 26.5%


## 7. O pipeline

In [14]:
pipeline = build_pipeline()
pipeline

C:\Users\Zeus\Projetos\Estudos\Pós MLE - FIAP\Tech Challenge - Grupo 57\churn-prediction\.venv\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:420: UserWarning: Unknown section O Que Faz
  self[section] = content


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('estrutural', ...), ('descarte', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,criar_delta_cobranca,False
,criar_flags,True
,colunas,"['services_quarter', 'status_quarter', ...]"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :t

In [15]:
pipeline.fit(X_treino, y_treino)

matriz_treino = pipeline.transform(X_treino)
matriz_teste = pipeline.transform(X_teste)

print(f"Entrada:  {X_treino.shape[1]} colunas")
print(f"Descarte: {len(pipeline.named_steps['descarte'].colunas_removidas_)} removidas")
print(f"Saida:    {matriz_treino.shape[1]} features apos one-hot")
print()
print(f"Matriz de treino: {matriz_treino.shape}")
print(f"Matriz de teste:  {matriz_teste.shape}")
print(f"Sem NaN/inf: {np.isfinite(matriz_treino).all() and np.isfinite(matriz_teste).all()}")

Entrada:  60 colunas


Descarte: 32 removidas
Saida:    52 features apos one-hot

Matriz de treino: (5634, 52)
Matriz de teste:  (1409, 52)
Sem NaN/inf: True


In [16]:
nomes = pipeline.named_steps["preprocessamento"].get_feature_names_out()
print(f"{len(nomes)} features finais. Amostra:")
for n in list(nomes)[:25]:
    print("   ", n)
print("    ...")

52 features finais. Amostra:
    num__demographics_age
    num__demographics_number_of_dependents
    num__services_number_of_referrals
    num__services_tenure_in_months
    num__services_avg_monthly_long_distance_charges
    num__services_avg_monthly_gb_download
    num__services_monthly_charge
    num__services_total_charges
    num__services_total_refunds
    num__services_total_extra_data_charges
    num__flag_sem_internet
    num__flag_sem_telefone
    cat__demographics_gender_Female
    cat__demographics_gender_Male
    cat__demographics_senior_citizen_No
    cat__demographics_senior_citizen_Yes
    cat__demographics_married_No
    cat__demographics_married_Yes
    cat__demographics_dependents_No
    cat__demographics_dependents_Yes
    cat__services_phone_service_No
    cat__services_phone_service_Yes
    cat__services_multiple_lines_No
    cat__services_multiple_lines_Yes
    cat__services_internet_type_Cable
    ...


Conferência de que o `fit` ficou restrito ao treino: se as estatísticas do scaler
coincidissem com as da base inteira, seria sinal de ajuste fora do fold.

In [17]:
def medias_do_scaler(pipe):
    return (
        pipe.named_steps["preprocessamento"].named_transformers_["num"].named_steps["escala"].mean_
    )


pipe_completo = build_pipeline().fit(X, y)

iguais = np.allclose(medias_do_scaler(pipeline), medias_do_scaler(pipe_completo))
print(f"Medias do treino == medias da base inteira? {iguais}")
print("OK, o fit ficou restrito ao treino." if not iguais else "ALERTA, revisar.")

Medias do treino == medias da base inteira? False
OK, o fit ficou restrito ao treino.


## 8. `delta_cobranca`: hipótese testada e descartada

A seção 12 da EDA calculou a diferença entre `total_charges` e `monthly_charge × tenure`,
verificou que não é zero e parou aí.

A hipótese era que essa diferença fosse o resíduo acumulado de reajustes e mudanças de
plano ao longo da relação, e que reajuste para cima funcionasse como gatilho de
cancelamento. Nenhuma outra coluna captura trajetória de preço: `monthly_charge` é o valor
de hoje e `total_charges` é o acumulado histórico.

Não se sustentou.

In [18]:
enriquecido = EngenhariaEstrutural(criar_delta_cobranca=True).fit_transform(modelagem)

print(enriquecido["delta_cobranca"].describe().round(2).to_string())
print()
correlacoes = (
    enriquecido[["delta_cobranca", "delta_cobranca_pct", cfg.ALVO]]
    .corr(numeric_only=True)[cfg.ALVO]
    .drop(cfg.ALVO)
    .round(3)
)
print("Correlacao com o alvo:")
print(correlacoes.to_string())

count    7043.00
mean        0.15
std        67.20
min      -370.85
25%       -28.60
50%         0.00
75%        28.50
max       373.25

Correlacao com o alvo:
delta_cobranca        0.000
delta_cobranca_pct   -0.012


Correlação de 0,00 e -0,01, contra o critério de corte de 0,05 definido antes de medir. A
mediana da diferença é exatamente zero, ou seja, `total_charges` bate com
`monthly_charge × tenure` para a maioria dos clientes: esta base não registra histórico de
reajuste.

A feature fica desligada por padrão em `build_pipeline`, e o parâmetro
`criar_delta_cobranca=True` continua disponível para quem quiser reproduzir o teste. Fica
registrado para que ninguém do grupo gaste tempo com a mesma ideia.

## 9. Baseline e a armadilha da acurácia

O bloco 6 do ML Canvas, que define a métrica de avaliação, está em branco.

In [19]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

candidatos = {
    "baseline (classe majoritaria)": DummyClassifier(strategy="most_frequent"),
    "regressao logistica": LogisticRegression(max_iter=1000, random_state=SEED),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
}

resultados = []
for nome, estimador in candidatos.items():
    pipe = build_pipeline(estimador).fit(X_treino, y_treino)
    pred = pipe.predict(X_teste)
    proba = pipe.predict_proba(X_teste)[:, 1]
    resultados.append(
        {
            "modelo": nome,
            "acuracia": (pred == y_teste).mean(),
            "roc_auc": roc_auc_score(y_teste, proba),
            "pr_auc": average_precision_score(y_teste, proba),
        }
    )

pd.DataFrame(resultados).round(4).set_index("modelo")

,acuracia,roc_auc,pr_auc
modelo,,,
baseline (classe majoritaria),0.7346,0.5000,0.2654
regressao logistica,0.8559,0.9127,0.7800
random forest,0.8495,0.9118,0.7861


A linha do baseline resume o problema: acurácia alta, ROC-AUC 0,5 e PR-AUC igual à
prevalência. É o modelo que não identifica ninguém.

Proposta de métrica para o bloco 6 do Canvas:

1. PR-AUC como métrica primária, por ser insensível ao desbalanceamento e focada na classe
   positiva.
2. ROC-AUC como secundária, para comparar com o 0,94 do `churn_score`.
3. Recall na classe positiva como restrição operacional. Churner não identificado é
   cliente perdido; falso positivo é um contato de retenção desnecessário. O custo é
   assimétrico e o threshold precisa refletir isso, o que é o bloco 8 do Canvas, também
   em branco.
4. Acurácia só entra em relatório acompanhada do baseline de 73,5%.

Nenhum destes modelos foi tunado e o `MLPClassifier`, que é o principal segundo o ADR-004,
ainda não entrou. Os números abaixo são piso, não resultado final.

In [20]:
melhor = build_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
).fit(X_treino, y_treino)

print(
    classification_report(
        y_teste,
        melhor.predict(X_teste),
        digits=3,
        target_names=["ficou", "cancelou"],
    )
)

              precision    recall  f1-score   support

       ficou      0.936     0.793     0.859      1035
    cancelou      0.598     0.850     0.702       374

    accuracy                          0.808      1409
   macro avg      0.767     0.822     0.780      1409
weighted avg      0.846     0.808     0.817      1409



`class_weight="balanced"` troca precisão por recall na classe positiva, que é o
comportamento desejado dado o custo assimétrico. Fica como referência para a discussão de
threshold.

## 10. Decisões e pendências

Decisões tomadas nesta etapa, para revisão do grupo:

1. Remover `status_churn_label` e `status_customer_status`, reescritas literais do alvo.
   Não estavam marcadas no Contrato 1 nem no `eda-findings.md`.
2. Remover as cinco colunas de vazamento identificadas pela EDA.
3. Usar `status_churn_score` como benchmark a superar (ROC-AUC 0,94), nunca como input.
4. Manter os 454 clientes `Joined` no treino. A hipótese de censura foi testada e o custo
   de removê-los (separação perfeita na faixa tenure ≤ 3, que responde por 31,9% dos
   churns) supera o benefício. Reversível com `filtrar_censura(df, remover_joined=True)`.
5. Deixar `services_offer` de fora até que se confirme quando a oferta é atribuída.
   Reversível com `build_pipeline(incluir_offer=True)`.
6. Deixar a geografia de fora. A base é 100% Califórnia, `locations_city` tem 1106 valores
   e as células ficam pequenas demais. Reversível com
   `build_pipeline(incluir_geografia=True)`.
7. Converter nulos legítimos em categoria explícita em vez de imputar ou descartar.
8. Separar zeros estruturais em flag própria, preservando o valor numérico de quem tem o
   serviço.
9. Descartar `delta_cobranca` após medir correlação de 0,00 com o alvo.
10. Adotar PR-AUC como métrica primária.

Pendências que dependem de decisão do grupo:

1. Janela temporal do alvo. `status_churn_value` significa "cancelou em qual período"? A
   base é snapshot de Q3. Sem essa definição o `/predict` devolve uma probabilidade sem
   unidade de tempo, e a decisão 4 acima muda se o alvo virar uma janela futura.
2. O Contrato 3, que descreve a API, usa nomes de coluna do CSV do Kaggle de 21 colunas,
   que é o dataset que o ADR-003 decidiu não usar. Quem estiver escrevendo os schemas
   Pydantic agora vai ter que refazer.
3. `docs/monitoring_plan.md` é citado no ML Canvas e no ADR-004 e não existe em nenhuma
   branch.
4. Limitação a registrar no Model Card: as decisões de feature vieram de uma EDA rodada
   sobre 100% dos dados, antes de qualquer split. É prática comum em análise descritiva,
   mas precisa estar documentada.